In [1]:
cd ..

/home/jim/bet


In [2]:

from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity
from prepare_data_inference import prepare_num_data_inference
from pyspark.ml.functions import vector_to_array
import mlflow 
import numpy as np
from mlflow.tracking import MlflowClient


In [3]:
def compute_metrics(df, threshold):
    pred = df.withColumn(
        "pred_label",
        (F.col("p_churn") >= threshold).cast("int")
    )
    metrics = pred.groupBy("next_7d_churn_idx", "pred_label").count()
    tp = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 1").select("count").first()
    fp = metrics.filter("next_7d_churn_idx = 0 AND pred_label = 1").select("count").first()
    fn = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 0").select("count").first()

    tp = tp[0] if tp else 0
    fp = fp[0] if fp else 0
    fn = fn[0] if fn else 0

    precision = float(np.round(tp / (tp + fp) ,2)) if (tp + fp) > 0 else 0.0
    recall = float(np.round(tp / (tp + fn), 2)) if (tp + fn) > 0 else 0.0
    f1 = (float(np.round( 2 * precision * recall / (precision + recall) , 2))
        if (precision + recall) > 0 else 0.0
    )

    return precision, recall, f1

In [4]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/13 23:18:43 WARN Utils: Your hostname, jim-VMware, resolves to a loopback address: 127.0.1.1; using 192.168.182.130 instead (on interface ens33)
26/03/13 23:18:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 23:18:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')

## Prepare date of a specific date for inference

In [6]:
test_date = '2024-05-10'
num_data_inference = prepare_num_data_inference(test_date) ## Only transactions features should be printed
num_data_inference.show(3)

failed_withdrawals_30d
deposit_count_30d
withdrawal_count_30d
withdrawal_ratio
+----------+--------------------+---------------------+------------------+------------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|player_idx|net_amount_result_7d|net_amount_result_30d|   balance_30d_ago|    balance_7d_ago|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|
+----------+--------------------+---------------------+------------------+------------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|        29|   798.4000000000001|   1687.6699999999

### Check consistency with training data (structure)

In [7]:
def compare(df1,df2): 
   assert (( df1.exceptAll(df2).count() == 0) & (df2.exceptAll(df1).count() == 0))

m1 = player_behavior.filter(F.col('reference_date')==test_date).select('player_idx').join(num_data_inference, how='inner', on='player_idx', )
compare(m1, num_data_inference)

### Load model

In [ ]:
mlflow.set_experiment("batch-mode experiment")
#mlflow.set_tracking_uri("file:./mlruns")

loaded_model = mlflow.spark.load_model(
    "models:/SparkLogisticRegression_train@production"
)

client = MlflowClient()
model_version = client.get_model_version_by_alias( name="SparkLogisticRegression_train",  alias="production")
run_id = model_version.run_id
run = mlflow.get_run(run_id)
print(run_id)

2026/03/13 23:22:25 INFO mlflow.spark: URI 'models:/SparkLogisticRegression_train@production/sparkml' does not point to the current DFS.
2026/03/13 23:22:25 INFO mlflow.spark: File 'models:/SparkLogisticRegression_train@production/sparkml' not found on DFS. Will attempt to upload the file.


d7ca5044cd984d25a887b09000b71087


In [68]:
def result_display(preds):
    preds = preds.select('player_idx','reference_date', 'p_churn', 'prediction')
    preds = preds.withColumn('risk_level', 
        F.when(F.col('p_churn')>=0.8, 'High')
        .when(F.col('p_churn')>=0.6, 'Medium')
        .when(F.col('p_churn')>=0.4, 'Low')
        .otherwise(F.lit('None')))
    flagged_players = preds.filter(F.col('prediction')== 1).select('player_idx', 'p_churn')
    return preds, flagged_players


results, flagged_players = result_display(test_preds)
results.show(4)

+----------+--------------+-------------------+----------+----------+
|player_idx|reference_date|            p_churn|prediction|risk_level|
+----------+--------------+-------------------+----------+----------+
|       506|    2024-05-10| 0.0583299746682745|       0.0|      None|
|       684|    2024-05-10|0.22118949241767105|       0.0|      None|
|       729|    2024-05-10|  0.422557289850085|       0.0|       Low|
|       745|    2024-05-10|0.22726606145282124|       0.0|      None|
+----------+--------------+-------------------+----------+----------+
only showing top 4 rows


In [ ]:
data_inference_ml = (player_behavior.select('player_idx','reference_date').filter(F.col('reference_date')==test_date)
.join(num_data_inference, how ='inner', on='player_idx')
.join(player_snapshot.select('player_idx', 'country', 'age_bucket'), on="player_idx", how="inner")
)

with mlflow.start_run(run_name=test_date):
    test_preds = (loaded_model.transform(data_inference_ml)
    .withColumn("p_churn", F.round(vector_to_array("probability")[1],2)))
    results, flagged_players = result_display(test_preds)
    mlflow.log_param("run_date", test_date)                # the inference date
    mlflow.log_param("run_id", run_id)                # the inference date
    mlflow.log_param("model", model_version.source)                # the inference date
    mlflow.log_param("threshold", threshold)          # threshold used for labeling
    mlflow.log_param("num_players", data_inference_ml.count())
    mlflow.log_param("num_flagged_players", flagged_players.count())
    mlflow.log_table(flagged_players.toPandas(), 'flagged_players.json')

In [93]:
run.data

<RunData: metrics={'upr': 0.5780944458731299}, params={'class_weight': '8.376801322938814',
 'elasticParam': '0.5',
 'git_branch': 'corrected-dates',
 'git_commit': '32cff6bd323258fb2adb4a07766680d3df1bc48b',
 'num_churn_train': '4233',
 'num_features': '16',
 'num_nonchurn_train': '35459',
 'platform': 'Linux-6.17.0-14-generic-x86_64-with-glibc2.42',
 'python_version': '3.13.7 (main, Mar  3 2026, 12:19:54) [GCC 15.2.0]',
 'regParam': '0.01',
 'test_end': '2024-06-22',
 'test_rows': '14494',
 'test_start': '2024-06-01',
 'threshold': '0.65',
 'train_end': '2024-05-31',
 'train_rows': '39692',
 'train_start': '2024-02-01'}, tags={'mlflow.log-model.history': '[{"run_id": "d7ca5044cd984d25a887b09000b71087", '
                             '"artifact_path": "spark_model", '
                             '"utc_time_created": "2026-03-13 '
                             '20:48:38.612772", "model_uuid": '
                             '"fd431dd1334e4c3c9d1bf732ae9e15d9", "flavors": '
             

## Apply backtest on test set

In [ ]:
test_start = run.data.params["test_start"]
test_end = run.data.params["test_end"]
threshold = run.data.params["threshold"]

test_df =  (player_behavior.filter(F.col('reference_date')>=test_start)
        .join(player_snapshot.select('player_idx', 'country', 'age_bucket'), on="player_idx", how="left")
        .join(labels, on=["player_idx", "reference_date"], how="inner")
         .withColumn("next_7d_churn_idx", F.col("next_7d_churn").cast("int")))

print(test_start, test_end)


2024-06-01 2024-06-22


In [ ]:
dates = (
    test_df
    .select("reference_date")
    .distinct()
    .orderBy("reference_date")
    .collect()
)

dates = [r.reference_date for r in dates]
results = []
for d in dates:
    daily_df = test_df.filter(F.col("reference_date") == d)
    preds = (loaded_model.transform(daily_df)
        .withColumn("p_churn", vector_to_array("probability")[1]))
    preds = preds.withColumn('risk_level', 
        F.when(F.col('p_churn')>=0.8, 'High')
        .when(F.col('p_churn')>=0.6, 'Medium')
        .when(F.col('p_churn')>=0.4, 'Low')
        .otherwise(F.lit('None')))

    precision, recall, f1 = compute_metrics(preds, threshold)

    results.append({
        "date": str(d),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "num_players": preds.count(),
        "num_flagged": preds.filter(F.col("prediction") == 1).count(),
        "num_churned": preds.filter(F.col("next_7d_churn_idx") == 1).count(),
        "num_churned_high_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'High')).count(),
        "num_churned_med_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'Medium')).count(),
        "num_churned_low_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'Low')).count() ,
        "num_churned_no_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'None')).count() ,
    })

In [48]:
backtest_df = spark.createDataFrame(results)
backtest_df.orderBy("date").show(5)

+----------+----+-----------+---------------------+--------------------+--------------------+-------------------+-----------+-----------+---------+------+
|      date|  f1|num_churned|num_churned_high_risk|num_churned_low_risk|num_churned_med_risk|num_churned_no_risk|num_flagged|num_players|precision|recall|
+----------+----+-----------+---------------------+--------------------+--------------------+-------------------+-----------+-----------+---------+------+
|2024-06-01|0.61|         59|                   38|                   5|                   5|                 11|         82|        657|     0.52|  0.73|
|2024-06-02|0.64|         64|                   40|                   6|                   4|                 14|         74|        658|     0.59|  0.69|
|2024-06-03|0.66|         68|                   43|                   6|                   4|                 15|         71|        658|     0.65|  0.68|
|2024-06-04|0.64|         71|                   42|                   

In [53]:
back_test_preds = (loaded_model.transform(test_df)
        .withColumn("p_churn", vector_to_array("probability")[1])
        .withColumn('risk_level', 
                F.when(F.col('p_churn')>=0.8, 'High')
                .when(F.col('p_churn')>=0.6, 'Medium')
                .when(F.col('p_churn')>=0.4, 'Low')
                .otherwise(F.lit('None')))
                )
back_test_preds.show(4)

+----------+--------------+-----------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+-------+----------+-------------+-----------------+-----------+--------------+-------------+--------------+--------------------+-----------------------+--------------------+--------------------+--------------------+----------+-------------------+----------+
|player_idx|reference_date|   balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|country|age_bucket|next_7d_churn|next_7d_churn_idx|country_idx|age_bucket_idx|  country_ohe|age_bucket_ohe|    numeric_features|numeric_

In [ ]:
pred_per_day = (back_test_preds.groupBy('reference_date')
    .agg(
        F.count('player_idx').alias('num_players'),
        F.sum(F.when(F.col("prediction") == 1, 1).otherwise(F.lit(0))).alias("num_flagged"),
        F.sum(F.when(F.col("next_7d_churn_idx") == 1, 1).otherwise(F.lit(0))).alias("num_churned"),
        F.sum(F.when(((F.col("next_7d_churn_idx") == 1) & (F.col("prediction") == 1)), 1).otherwise(F.lit(0))).alias("tp"),
        F.sum(F.when(((F.col("next_7d_churn_idx") == 1) & (F.col("prediction") == 0)), 1).otherwise(F.lit(0))).alias("fn"),
        F.sum(F.when(((F.col("next_7d_churn_idx") == 0) & (F.col("prediction") == 0)), 1).otherwise(F.lit(0))).alias("tn"),
        F.sum(F.when(((F.col("next_7d_churn_idx") == 0) & (F.col("prediction") == 1)), 1).otherwise(F.lit(0))).alias("fp"), 
        F.sum(F.when(((F.col("next_7d_churn_idx") == 1) & (F.col("risk_level") == 'High')), 1).otherwise(F.lit(0))).alias("num_churned_high_risk"), 
        F.sum(F.when(((F.col("next_7d_churn_idx") == 1) & (F.col("risk_level") == 'Medium')), 1).otherwise(F.lit(0))).alias("num_churned_med_risk"), 
        F.sum(F.when(((F.col("next_7d_churn_idx") == 1) & (F.col("risk_level") == 'Low')), 1).otherwise(F.lit(0))).alias("num_churned_low_risk"), 
        F.sum(F.when(((F.col("next_7d_churn_idx") == 1) & (F.col("risk_level") == 'None')), 1).otherwise(F.lit(0))).alias("num_churned_no_risk"), 
       )
    .withColumn('num_churned', F.col('tp') + F.col('fn'))
    .withColumn('precision', F.when(F.col('tp') + F.col('fp') > 0, F.round(F.col('tp') / (F.col('tp')+ F.col('fp')) ,2)).otherwise(F.lit(0)))
    .withColumn('recall', F.when(F.col('tp') + F.col('fn') > 0, F.round(F.col('tp') / (F.col('tp')+ F.col('fn')) ,2)).otherwise(F.lit(0)))
    .withColumn('f1', F.when(F.col('precision') + F.col('recall') > 0, F.round(2 * F.col('precision') * F.col('recall') / (F.col('precision')+ F.col('recall')) ,2)).otherwise(F.lit(0)))
    .withColumn('churned_rate_high_risk',F.round(F.col('num_churned_high_risk') / F.col('num_churned'),2))
    .withColumn('churned_rate_med_risk', F.round(F.col('num_churned_med_risk') / F.col('num_churned'),2))
    .withColumn('churned_rate_low_risk', F.round(F.col('num_churned_low_risk') / F.col('num_churned'),2))
    .withColumn('churned_rate_no_risk', F.round(F.col('num_churned_no_risk') / F.col('num_churned'),2))
.drop('tp', 'fn', 'tn','fp')

).orderBy("reference_date")
    
pred_per_day.show(8)


+--------------+-----------+-----------+-----------+---------------------+--------------------+--------------------+-------------------+---------+------+----+----------------------+---------------------+---------------------+--------------------+
|reference_date|num_players|num_flagged|num_churned|num_churned_high_risk|num_churned_med_risk|num_churned_low_risk|num_churned_no_risk|precision|recall|  f1|churned_rate_high_risk|churned_rate_med_risk|churned_rate_low_risk|churned_rate_no_risk|
+--------------+-----------+-----------+-----------+---------------------+--------------------+--------------------+-------------------+---------+------+----+----------------------+---------------------+---------------------+--------------------+
|    2024-06-01|        657|         82|         59|                   38|                   5|                   5|                 11|     0.52|  0.73|0.61|                  0.64|                 0.08|                 0.08|                0.19|
|    2024-06

In [94]:
pred_per_day.columns

['reference_date',
 'num_players',
 'num_flagged',
 'num_churned',
 'num_churned_high_risk',
 'num_churned_med_risk',
 'num_churned_low_risk',
 'num_churned_no_risk',
 'precision',
 'recall',
 'f1',
 'churned_rate_high_risk',
 'churned_rate_med_risk',
 'churned_rate_low_risk',
 'churned_rate_no_risk']

In [ ]:
precision, recall, f1 = compute_metrics(back_test_preds, threshold)
f1



0.6

In [ ]:
mlflow.log_metrics


TypeError: cannot convert the series to <class 'float'>

In [95]:
select_cols = ['precision',
 'recall',
 'f1',
 'churned_rate_high_risk',
 'churned_rate_med_risk',
 'churned_rate_low_risk',
 'churned_rate_no_risk']

df_sum = pred_per_day.select([F.avg(c).alias('avg_'+ c) for c in select_cols])
df_sum.show()

+------------------+------------------+------------------+--------------------------+-------------------------+-------------------------+------------------------+
|     avg_precision|        avg_recall|            avg_f1|avg_churned_rate_high_risk|avg_churned_rate_med_risk|avg_churned_rate_low_risk|avg_churned_rate_no_risk|
+------------------+------------------+------------------+--------------------------+-------------------------+-------------------------+------------------------+
|0.6145454545454546|0.5995454545454545|0.6031818181818183|        0.4986363636363636|      0.14272727272727276|      0.13772727272727273|     0.22227272727272734|
+------------------+------------------+------------------+--------------------------+-------------------------+-------------------------+------------------------+



## Probability Calibration (model reliability)

In [59]:
preds = preds.withColumn(
    "prob_bin",
    F.floor(F.col("p_churn") * 10) / 10
)

calibration = (
    preds
    .groupBy("prob_bin")
    .agg(
        F.avg("next_7d_churn_idx").alias("actual_rate"),
        F.count("*").alias("players")
    )
    .orderBy("prob_bin")
)
calibration.show()

+--------+--------------------+-------+
|prob_bin|         actual_rate|players|
+--------+--------------------+-------+
|     0.0|0.026777875329236173|   2278|
|     0.1| 0.04144282425172678|   2606|
|     0.2| 0.04505994212484498|   2419|
|     0.3| 0.04951923076923077|   2080|
|     0.4| 0.06819452207937395|   1789|
|     0.5| 0.10059171597633136|   1183|
|     0.6| 0.16962025316455695|    790|
|     0.7|  0.3382352941176471|    340|
|     0.8|  0.8366733466933868|    998|
|     0.9|  0.9090909090909091|     11|
+--------+--------------------+-------+



### Coclusions 
- The model is well ranked (higher probability → higher churn).

- A very high-risk segment (>0.8) exists with ~84% churn.

- Medium probabilities slightly overestimate churn.

- The model is very useful for targeted retention campaigns.

- A threshold around 0.6–0.75 would likely be optimal depending on recall needs.